# Data Processing

After scraping, the raw files still need to be turned into the format used by the forecasting model. The main issue is the weather data: the model should not be trained on perfect future weather observations, because those values would not be known when making a real forecast.

Instead, the processing step creates forecast-like weather inputs. It uses the real previous-run weather forecasts to estimate typical forecast errors, then applies those errors to the longer historical weather series. This gives us weather predictors that look like forecasts across the full modelling period.

| Processing output | What it contains |
|-------------------|------------------|
| `data/weather_error_distributions.csv` | Estimated weather forecast errors by variable and forecast horizon |
| `data/weather_errors_raw.csv` | Raw forecast error observations used to calculate the distributions |
| `data/forecast_dataset.parquet` | Forecast-like weather dataset with issue times, target times, horizons, and time features |
| `data/model_dataset.parquet` | Model-ready table with target prices, issue-time price lags, and wind-direction sin/cos features |
| `data/weather_dk1_dashboard.parquet` | Smaller DK1 weather file used for plotting and inspection |

The full processing step is run with one command. It first builds the weather error distributions, then creates the forecast-like weather dataset, and finally adds the price target and issue-time price lags used by the model.

In [ ]:
!python -m src.data.data_processing

The plot below shows the estimated weather forecast error distributions across forecast horizons. It is used as a sanity check: errors should generally become wider as the horizon increases, because longer-range forecasts are more uncertain.

In [ ]:
from src.data.data_processing import plot_weather_error_distributions

plot_weather_error_distributions()

The next plot shows the processed weather data in the same style as the dashboard: historical actual weather first, followed by the 5-day forecast window. The dotted line is the generated forecast, and the actual line in the forecast window is shown so the reader can compare forecast-like inputs with what happened afterwards.

Change `issue_time` to inspect a different forecast period.

In [ ]:
from src.data.data_processing import plot_weather_forecast_dashboard_style

plot_weather_forecast_dashboard_style(
    issue_time="2026-04-23 00:00",
    ctx_days=7,
    show_actuals=True,
)

The file that moves forward to the forecasting model is now `data/model_dataset.parquet`. It already contains the forecast-like weather predictors, the matching electricity price target, and price lags based on information available at `issue_time`.